# Build a chord-preference labeling sheet

Run notebook 03 first. This notebook estimates the key, proposes a compact candidate set, and exports a CSV. The diagnostic ordering only makes labeling easier; it is not the learned recommendation.

In [1]:
import os
from pathlib import Path
import pandas as pd

from reharmonize_models import (
    aggregate_melody_by_chord, create_candidate_sheet, estimate_key, melody_pitch_track,
)

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
audio_setting = os.environ.get('AUDIO_PATH')
AUDIO = Path(audio_setting) if audio_setting else ROOT / 'data/song.mp3'
if not AUDIO.is_absolute():
    AUDIO = ROOT / AUDIO
DURATION = float(os.environ.get('ANALYSIS_DURATION', '30'))
VOCALS = ROOT / 'artifacts' / f'{AUDIO.stem}-vocals.wav'
CHORDS = ROOT / 'artifacts' / f'{AUDIO.stem}-smoothed.lab'
PREFERENCES = ROOT / 'data' / f'{AUDIO.stem}-preferences.csv'
AUDIO, VOCALS, CHORDS

(PosixPath('/Users/danendra/Documents/ADA26/C6/Models/data/song.mp3'),
 PosixPath('/Users/danendra/Documents/ADA26/C6/Models/artifacts/song-vocals.wav'),
 PosixPath('/Users/danendra/Documents/ADA26/C6/Models/artifacts/song-smoothed.lab'))

In [2]:
if not VOCALS.exists() or not CHORDS.exists():
    raise FileNotFoundError('Run notebooks/03_vocal_aware_analysis.ipynb first')
chords = pd.read_csv(CHORDS, sep=r'\s+', names=['start', 'end', 'chord'])
melody = melody_pitch_track(VOCALS, duration=DURATION)
segments = aggregate_melody_by_chord(chords, melody)
key = estimate_key(chords)
print({'estimated_key': key, 'segments': len(segments)})

{'estimated_key': 'E:maj', 'segments': 15}


In [3]:
sheet = create_candidate_sheet(segments, key=key, max_candidates=8)
PREFERENCES.parent.mkdir(parents=True, exist_ok=True)
sheet.to_csv(PREFERENCES, index=False)
print(f'Saved {len(sheet)} rows to {PREFERENCES}')
sheet.head(16)

Saved 112 rows to /Users/danendra/Documents/ADA26/C6/Models/data/song-preferences.csv


,segment_id,start,end,key,previous_chord,source_chord,next_chord,candidate,display_order,melody_pc_0,...,melody_pc_8,melody_pc_9,melody_pc_10,melody_pc_11,chord_tone_ratio,semitone_proximity_ratio,extension_ratio,voiced_fraction,evidence,preferred
0,1,1.111,6.389,E:maj,N,E,A,E:7,1,0.0,...,0.0,0.0,0.0,0.0,0.583778,0.416222,0.000000,0.009473,0.25,
1,1,1.111,6.389,E:maj,N,E,A,A#:7,2,0.0,...,0.0,0.0,0.0,0.0,0.583778,0.416222,0.000000,0.009473,0.25,
2,1,1.111,6.389,E:maj,N,E,A,E:maj7,3,0.0,...,0.0,0.0,0.0,0.0,0.416222,0.583778,0.000000,0.009473,0.25,
3,1,1.111,6.389,E:maj,N,E,A,G#:min,4,0.0,...,0.0,0.0,0.0,0.0,0.416222,0.583778,0.000000,0.009473,0.25,
4,1,1.111,6.389,E:maj,N,E,A,G#:min7,5,0.0,...,0.0,0.0,0.0,0.0,0.416222,0.583778,0.000000,0.009473,0.25,
5,1,1.111,6.389,E:maj,N,E,A,B,6,0.0,...,0.0,0.0,0.0,0.0,0.416222,0.583778,0.000000,0.009473,0.25,
6,1,1.111,6.389,E:maj,N,E,A,G#:7,7,0.0,...,0.0,0.0,0.0,0.0,0.416222,0.583778,0.000000,0.009473,0.25,
7,1,1.111,6.389,E:maj,N,E,A,E,8,0.0,...,0.0,0.0,0.0,0.0,0.000000,0.416222,0.583778,0.009473,0.25,
8,2,6.389,8.333,E:maj,E,A,E,A,1,0.0,...,0.0,1.0,0.0,0.0,1.000000,0.000000,0.000000,0.005144,0.05,
9,2,6.389,8.333,E:maj,E,A,E,F#:min7,2,0.0,...,0.0,1.0,0.0,0.0,1.000000,0.000000,0.000000,0.005144,0.05,


## Label the candidates

Open the exported CSV in Numbers or a spreadsheet editor. For every `segment_id`, listen to that time range and set `preferred` to `1` for exactly one candidate. Set the other candidates in that segment to `0`. Keep the CSV under `data/`; audio and private labels are ignored by Git.

In [4]:
sheet.groupby('segment_id')[['start', 'end', 'source_chord']].first()

,start,end,source_chord
segment_id,,,
1,1.111,6.389,E
2,6.389,8.333,A
3,8.333,12.500,E
4,12.500,14.074,E:7
5,14.074,16.019,A
6,16.019,19.259,E
7,19.259,20.000,B
8,20.000,21.574,E:7
9,21.574,23.426,A
